# Part A: Explore Dataset. Missing reports, plot images. Front and Lateral Views of the lungs


In [ ]:
from pathlib import Path

In [ ]:
Path.cwd() #current working directory
#You could write paths as strings like "..\\data\\raw", but that gets messy: Windows uses \, Linux and Colab use /, 
# and you'd be gluing strings together by hand. 
# A Path object knows it's a path and does this for you.
#The / operator joins pieces of a path. Here it doesn't mean division; Path gives / this meaning.

In [ ]:
DATA_RAW = Path("..")/"data"/"raw" #DATA_RAW is in capital letters cause its the convention that it is not changing
DATA_RAW

In [ ]:
print(DATA_RAW.resolve())

In [ ]:
REPORTS = DATA_RAW / "ecgen-radiology"
REPORTS

In [ ]:
files = sorted(REPORTS.glob("*.xml"))


In [ ]:
list(REPORTS.glob('?.xml'))

In [ ]:
list(REPORTS.glob('1*.xml'))[0:5]

In [ ]:
list(REPORTS.glob('1?.xml'))

In [ ]:
for file in files[:5]:
    print(file)
#put 10.xml right after 1.xml, and 100.xml after that. 
# That's because it sorts as text, character by character, like words in a dictionary. 
# "10" comes before "2" because "1" comes before "2". 
# For counting this doesn't matter,

In [ ]:
f  = files[0]
f.name, f.suffix

In [ ]:
IMAGES = DATA_RAW / "images"
images = sorted(IMAGES.glob('*.png'))

In [ ]:
print(f'Reports = {len(files)} and Images = {len(images)}')

In [ ]:
image_folder = sorted(IMAGES.glob('*'))
len(image_folder)

In [ ]:
for image in image_folder:
    if image.suffix != '.png':
        print(image)

In [ ]:
import xml.etree.ElementTree as ET

In [ ]:
toy = """<library> 
            <book id="b1">
                <title>Dune</title>
                <author>Herbert</author>
            </book>
            <book id="b2">
                <title>Emma</title>
                <author></author>
            </book>
        </library>"""
#triple quotes: lets a string span several lines

In [ ]:
root = ET.fromstring(toy)
print(root)
print(root.tag)

In [ ]:
for child in root:
    print(child.tag, child.attrib)

In [ ]:
first_book = root.find("book")
print(first_book.get("id"))
print(len(root.findall("book")))
print(first_book.find("title").text)

In [ ]:
print(root.find("title")) #searches for the first matching direct child, or None if there isn't one
print([t.text for t in root.iter("title")]) # find looks one level down; iter looks everywhere

In [ ]:
second_author = root.findall("book")[1].find("author")
print(second_author.text)

In [ ]:
first_report = REPORTS/'1.xml'
root = ET.parse(first_report).getroot()
root.tag

In [ ]:
print(root.find('uId').attrib)

In [ ]:
print([t.attrib.values() for t in root.iter("AbstractText")]) 

In [ ]:
print([t.text for t in root.iter("AbstractText")]) 

In [ ]:
for t in root.iter("AbstractText"):
    print(f'{t.attrib.get("Label")}: {t.text}')

In [ ]:
print([t.attrib.get('id') for t in root.findall('parentImage')])

In [ ]:
empty_xmls = []
for xml_file in sorted(REPORTS.glob("*.xml")):
    root =  ET.parse(xml_file).getroot()
    for label in root.iter("AbstractText"):
        if label.attrib.get('Label') == 'FINDINGS' and label.text is None:
            empty_xmls.append(xml_file)

In [ ]:
print(empty_xmls[0:5])

In [ ]:
len(empty_xmls)

The model that we want to build has to be trained on the findings sections. 13% of the dataset doesnt have a value in that field(530 out of 3955). The reports that can be used are 3,955 − 530 = 3,425. 

In [ ]:
#python imaging library, installed as pillow but import it as PIL
from PIL import Image

In [ ]:
img = Image.open(IMAGES / "CXR1_1_IM-0001-3001.png")
print(img.size)
print(img.mode)
#an image of course it is just a numpy array

In [ ]:
import numpy as np
arr = np.array(img)

In [ ]:
print(arr.shape)
print(arr.dtype, arr.min(), arr.max())
#The order flipped! PIL said (width, height) = (512, 624). numpy says (height, width, channels) = (624, 512, 3),
# because arrays are rows first, like a matrix. Same layout as the cat images in Course 1, which were (64, 64, 3).
#uint8 means an unsigned 8-bit integer: whole numbers from 0 to 255. That's why you divided images by 255 in the course, 
# to scale them to 0–1.
#Heads-up for later: PyTorch wants a third ordering, (channels, height, width). 
# We'll handle that in Phase 1. For now just notice that image libraries don't agree.

In [ ]:
np.array_equal(arr[:,:,1], arr[:,:,2]) 
#the three chanells are equal, the three different matrices.An X-ray has no colour.
# why might someone store a grayscale X-ray this way?

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
plt.imshow(arr[:,:,1])

In [ ]:
plt.imshow(arr[:,:,1], cmap="gray") #doesnt know its grayscale cause we feed a 2-D array(onnly give 1 channel)
#unless we give the wholem array(arr[:,:])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 5))

axes[0].imshow(arr)
axes[0].set_title("full RGB array")
axes[0].axis("off")

axes[1].imshow(arr[:, :, 0])
axes[1].set_title("channel 0")
axes[1].axis("off")

In [ ]:
first_report = REPORTS/'100.xml'
root = ET.parse(first_report).getroot()
parent_images = [t.attrib.get('id') for t in root.findall('parentImage')]


In [ ]:
report1_images = []
for image in parent_images:
    image_path = image + '.' + 'png'
    report1_images.append(Image.open(IMAGES / image_path))


In [ ]:
for img in report1_images:
    print(img.size, img.mode)

In [ ]:
n_images = len(report1_images)

if n_images == 0:
    print("This report has no images")
else:
    # squeeze=False: axes is always 2-D (rows, columns), even with a single image
    fig, axes = plt.subplots(1, n_images, figsize=(5 * n_images, 5), squeeze=False)

    for i, img in enumerate(report1_images):
        axes[0, i].imshow(img)
        axes[0, i].set_title(parent_images[i])
        axes[0, i].axis("off")

    plt.show()

We can see that some xlm files only have 1 parent image or maybe None. Also the two view have different sizes

# Part B: The inventory table


The goal: one table with one row per report, containing everything  explored in Part A. Then every inventory question becomes a short pandas command.
Nested dictionary:
reports = {
    "CXR1": {"comparison": "None.", "n_images": 2, ...},
    "CXR2": {"comparison": ...,     "n_images": 1, ...},
}
Or just a list:
records = [
    {"report_id": "CXR1", "comparison": "None.", "n_images": 2, ...},
    {"report_id": "CXR2", "comparison": ...,     "n_images": 1, ...},
]
The list seems to be a better way here cause it goes straight into pandas


In [ ]:
#start by extracting the values from '1.xml' into a dictionary
first_xlm =  REPORTS/'1002.xml'

In [ ]:
def parse_report(path):
    sections = {}
    root = ET.parse(path).getroot()

    #uId
    sections['report_id'] = root.find('uId').attrib.get('id')

    #AbstractText
    for label in root.iter('AbstractText'):
        sections[label.get('Label').lower()] = label.text

    #parentImages
    parent_images=[]
    for parent_image in root.iter('parentImage'):
        parent_images.append(parent_image.attrib.get('id'))
    sections["image_ids"] = parent_images
    sections["n_images"] = len(parent_images)

    #mesh
    mesh = root.find('MeSH')
    mesh_major = [m.text for m in mesh.findall("major")]
    mesh_automatic = [m.text for m in mesh.findall("automatic")]

    sections['mesh_major'] = mesh_major
    sections['mesh_automatic'] = mesh_automatic

    return sections

In [ ]:
sections = parse_report(first_xlm)

In [ ]:
sections

# Part C: Parse the files and create the dataset

In [ ]:
#Build the list of records
filepaths = list(REPORTS.glob("*.xml")) #join all the xml files into a list
filepaths[0:5]

In [ ]:
records = []

for filepath in filepaths:
    records.append(parse_report(filepath))
    
len(records)

In [ ]:
import pandas as pd

df = pd.DataFrame(records)

df.shape

In [ ]:
df[0:2]

In [ ]:
df["report_id"].duplicated().sum()

In [ ]:
abstract_text_df = df[['comparison','indication','findings','impression']]

In [ ]:
abstract_text_df.isna().sum()

Why all 4, if the model only learns to write Findings?

The other sections may matter later:
Impression is the radiologist's short conclusion.
When Findings is empty, Impression could be a fallback text to train on. That's a real option to weigh in Phase 3.
Indication is why the X-ray was taken (e.g. "chest pain"). It could become extra input to the model one day.
Comparison says whether there was an earlier X-ray. It's probably the least useful for us.

In [ ]:
df['n_images'].value_counts().sort_index()

In [ ]:
df['n_images'].sum() # equals the PNG count from Part A

In [ ]:
usable = df[df['findings'].notna() & (df['n_images']>0)]

In [ ]:
df['findings'].notna().sum()

In [ ]:
sample_df = usable.sample(5, random_state=42)

In [ ]:
for text in sample_df["findings"]:
    print(text)
    print("-" * 60)

In [ ]:
df.columns

In [ ]:
df['image_ids'][0:5]

In [ ]:
for report,image in zip(df['report_id'][0:5], df['image_ids'][0:5]):
    print(report,image)

In [ ]:
image_to_report = {}
for report_id, image_ids in zip(df['report_id'], df['image_ids']):
    for image_id in image_ids:
        if image_id not in image_to_report:
            image_to_report[image_id] = report_id
        else:
            print("duplicate:", image_id)  # an image claimed by two reports


In [ ]:
image_to_report

In [ ]:
print(len(image_to_report), df['n_images'].sum())

#they are equal so that means that every image is distinct and not repeated in other reports
#this says nothing about patients: one patient could have several reports, each with its own images.
#there is no patient ID in the XML -> open question for Step 0.2


In [ ]:
images = list(IMAGES.glob('*.png'))
len(images)

In [ ]:
image_records = []

for image_path in images:
    image_id = image_path.stem
    with Image.open(image_path) as img:
        width, height = img.size
        mode = img.mode

    image_records.append({
        "image_id": image_id,                          # from path
        "report_id": image_to_report.get(image_id),    # .get: an orphan image becomes None instead of crashing
        "width": width,
        "height": height,
        "mode": mode,
    })
img_df = pd.DataFrame(image_records)


In [ ]:
img_df

In [ ]:
img_df['mode'].value_counts()

In [ ]:
img_df.describe()
#count	how many non-missing values	7,470 in both columns, so no missing sizes
#mean	the average	width 512, height about 533
#std	standard deviation: how spread out the values are	width 0, height about 81
#min / max	smallest / largest	height ranges from 362 to 873
#25% / 50% / 75%	quartiles: 25% of values are below the first, 
# 50% below the second (the median), 75% below the third	height 484 / 512 / 624

std = 0 and min = max = 512 mean every single image is exactly 512 pixels wide. That can't be a coincidence with real X-rays from different machines. The provider resized every image to a width of 512, keeping the proportions. So the height is whatever the original shape produced, which is why heights vary and widths don't

In [ ]:
img_df['report_id'].isna().sum() #no orphan images. that they have no report id.

In [ ]:
#different image sizes
#take care of this diffrence. IMPORTANT

#img_df['width']	one column (a Series)
#img_df[['width', 'height']]	two columns (a smaller DataFrame)
#img_df['width']['height']	❌ the row labelled 'height' inside the width column
len(img_df[['width','height']].drop_duplicates())

In [ ]:
(img_df['height'] > img_df['width']).sum() #portrait images

In [ ]:
(img_df['height'] < img_df['width']).sum() #landscape images

In [ ]:
(img_df['height'] == img_df['width']).sum() #square images

In [ ]:
img_df["height"].value_counts().head(10)

In [ ]:
img_df["height"].hist(bins=50)

In [ ]:
portrait = img_df[img_df['height'] > img_df['width']]
landscape = img_df[img_df['height'] < img_df['width']]

portrait_samples = portrait.sample(3, random_state=0)
landscape_samples = landscape.sample(3,random_state=0)

In [ ]:
portrait_samples

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(12, 8))

# row 0: portrait samples
for col, image_id in enumerate(portrait_samples["image_id"]):
    img = Image.open(IMAGES / f"{image_id}.png")
    axes[0, col].imshow(img)
    axes[0, col].set_title(f"portrait\n{image_id}")
    axes[0, col].axis("off")

# row 1: landscape samples (same code, row 1 and the other table)
for col, image_id in enumerate(landscape_samples["image_id"]):
    img = Image.open(IMAGES / f"{image_id}.png")
    axes[1, col].imshow(img)
    axes[1, col].set_title(f"landscape\n{image_id}")
    axes[1, col].axis("off")

plt.tight_layout()
plt.show()
